In [ ]:
# Download libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import GradientBoostingClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
from tensorflow.python.client import device_lib
import gc
import warnings
import os
import json
import optuna
import shap

# Input data files
warnings.filterwarnings("ignore")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
def engineer_features(df: pd.DataFrame, fare_bins=None, age_edges=None, survival_map=None, te_maps: dict | None = None, global_mean: float | None = None) -> tuple[pd.DataFrame, np.ndarray, list]:
    out = df.copy()

    # Family features
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)

    # Title extraction and mapping
    out["Title"] = out["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    title_map = {
        "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs", "Lady": "Rare", "Countess": "Rare",
        "Capt": "Rare", "Col": "Rare", "Don": "Rare", "Dr": "Rare", "Major": "Rare",
        "Rev": "Rare", "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare"
    }
    out["Title_Type"] = out["Title"].replace(title_map)

    # Cabin & Ticket features
    out["CabinDeck"] = out["Cabin"].str[0].fillna("U")
    out["HasCabin"] = (~out["Cabin"].isna()).astype(int)
    out["TicketPrefix"] = (
        out["Ticket"]
        .str.replace(r"\d+", "", regex=True)
        .str.replace(".", "", regex=False)
        .str.strip()
        .replace("", "NUM")
    )
    out["TicketGroupSize"] = out.groupby("Ticket")["Ticket"].transform("count")

    # Fare per person
    out["FarePerPerson"] = (out["Fare"] / out["FamilySize"]).replace([np.inf, -np.inf], np.nan)
    out["FarePerPerson"] = out["FarePerPerson"].clip(upper=100)

    # Embarked imputation
    embarked_mode = out["Embarked"].mode(dropna=True)
    embarked_fill = embarked_mode.iloc[0] if not embarked_mode.empty else "S"
    out["Embarked"] = out["Embarked"].fillna(embarked_fill)

    # Fare imputation
    fare_grp = out.groupby(["Pclass", "Embarked"])["Fare"].transform("median")
    out["Fare"] = out["Fare"].fillna(fare_grp)

    # Age imputation
    age_medians = out.groupby(["Sex", "Pclass", "Title_Type"])["Age"].transform("median")
    out["Age"] = out["Age"].fillna(age_medians)

    # Child / Mother
    out["IsChild"] = (out["Age"] <= 12).astype(int)
    out["IsMother"] = (
        (out["Sex"] == "female") &
        (out["Parch"] > 0) &
        (out["Age"] >= 18) &
        (out["Title_Type"] == "Mrs")
    ).astype(int)

    # Interactions
    out["SexBin"] = (out["Sex"] == "male").astype(int)
    out["Age_x_Class"] = out["Age"] * out["Pclass"]
    out["Sex_x_Class"] = out["SexBin"] * out["Pclass"]
    out["HasSib"] = (out["SibSp"] > 0).astype(int)
    out["HasPar"] = (out["Parch"] > 0).astype(int)
    out["NameLen"] = out["Name"].str.len()

    # Fare binning
    if fare_bins is None:
        _, fare_bins = pd.qcut(out["Fare"], q=4, retbins=True, duplicates="drop")
    out["FareBin"] = pd.cut(out["Fare"], bins=fare_bins, include_lowest=True, labels=False)

    # Age binning
    if age_edges is None:
        age_edges = [0, 12, 20, 40, 60, 120]
    out["AgeBin"] = pd.cut(out["Age"], bins=age_edges, include_lowest=True, labels=False)

    # Frequency encoding for categorical features
    for col in ["TicketPrefix", "CabinDeck", "Title_Type", "Embarked"]:
        freq = out[col].value_counts(normalize=True)
        out[col + "_Freq"] = out[col].map(freq)

    # FamilySurvivalRate
    if survival_map is not None:
        out["FamilySurvivalRate"] = out["FamilySize"].map(survival_map)
    else:
        out["FamilySurvivalRate"] = np.nan

    
    # Target Encoding
    if te_maps is not None:
        # Only encode columns present in both the data and te_maps
        for col, mapping in te_maps.items():
            if col in out.columns:
                te_col = f"{col}_TE"
                out[te_col] = out[col].map(mapping).astype(float)
                # Fill unseen categories with global mean if provided
                if global_mean is not None:
                    out[te_col] = out[te_col].fillna(global_mean)


    # Fare and Age transformations
    out["Fare_log"] = np.log1p(out["Fare"])  # Log transform to reduce skew
    out["FarePerPerson_log"] = np.log1p(out["FarePerPerson"])
    out["Age_log"] = np.log1p(out["Age"])
    out["Age_squared"] = out["Age"] ** 2  # Polynomial feature
    out["Fare_squared"] = out["Fare"] ** 2

    # Drop unneeded columns
    drop_cols = ["Name", "Ticket", "Cabin", "Title"]
    out = out.drop(columns=[col for col in drop_cols if col in out.columns])

    return out, fare_bins, age_edges

In [ ]:
# Train / test
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test  = pd.read_csv("/kaggle/input/titanic/test.csv")

In [ ]:
# Random seed
np.random.seed(42)
tf.random.set_seed(42)

# Check GPUs
print(device_lib.list_local_devices())
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Select features
features = [
    # numeric
    "Pclass", "Age", "Fare", "FamilySize", "FarePerPerson", "IsAlone",
    "TicketGroupSize", "HasCabin", "NameLen", "IsChild", "IsMother",
    "Age_x_Class", "Sex_x_Class",  "TicketPrefix_Freq", "CabinDeck_Freq", 
    "Title_Type_Freq", "Embarked_Freq", "FamilySurvivalRate",
    "Fare_log", "FarePerPerson_log", "Age_log", "Age_squared", "Fare_squared",

    # target-encoded features
    "TicketPrefix_TE", "CabinDeck_TE", "Title_Type_TE", "Embarked_TE",

    # categorical
    "Sex", "Embarked", "Title_Type",
    "CabinDeck", "TicketPrefix", "FareBin", "AgeBin"
]

# Define target
target = "Survived"

# Build preprocessing (impute -> encode -> scale)
def make_preprocess():
    
    # Numeric features
    numeric_features = [
        "Pclass", "Age", "Fare", "FamilySize", "FarePerPerson",
        "IsAlone", "TicketGroupSize", "HasCabin", "NameLen",
        "IsChild","IsMother", "Age_x_Class", "Sex_x_Class",
        "TicketPrefix_Freq", "CabinDeck_Freq", "Title_Type_Freq", 
        "Embarked_Freq", "FamilySurvivalRate", "Fare_log", 
        "FarePerPerson_log", "Age_log", "Age_squared", "Fare_squared",

        # target-encoded features
        "TicketPrefix_TE", "CabinDeck_TE", "Title_Type_TE", "Embarked_TE"

    ]
    # Categorical features
    categorical_features = [
        "Sex", "Embarked", "Title_Type", "CabinDeck", "TicketPrefix",
        "FareBin", "AgeBin"
    ]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    # OneHotEncoder
    try:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    # Build categorical transformer
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", cat_encoder),
    ])

    # Transform
    pre = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )
    return pre

In [ ]:
# Optuna hyperparameter tuning
def objective(trial):
    
    # Hyperparameter search area
    n_estimators = trial.suggest_int("n_estimators", 300, 2000)
    learning_rate = trial.suggest_float("learning_rate", 0.005, 0.1, log=True)
    subsample = trial.suggest_float("subsample", 0.5, 1)
    max_depth = trial.suggest_int("max_depth", 2, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8, None])
    min_samples_split = trial.suggest_int("min_samples_split", 2, 50)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 2, 50)
    min_impurity_decrease = trial.suggest_float("min_impurity_decrease", 1e-8, 1e-2, log=True)  
    max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 8, 63)              
    validation_fraction = trial.suggest_float("validation_fraction", 0.1, 0.3)  
    n_iter_no_change = trial.suggest_int("n_iter_no_change", 10, 50)          
    tol = trial.suggest_float("tol", 1e-4, 1e-2, log=True)   

    # TE config
    te_cols = ["TicketPrefix", "CabinDeck", "Title_Type", "Embarked"]
    alpha = 20 

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    fare_bins, age_edges = None, None

    for fold_idx, (tr_idx, va_idx) in enumerate(cv.split(train, train[target]), start=1):
        # Feature engineering
        temp_train_fe, fare_bins, age_edges = engineer_features(
            train.iloc[tr_idx], fare_bins=fare_bins, age_edges=age_edges
        )

        survival_map = temp_train_fe.groupby("FamilySize")["Survived"].mean().to_dict()
        
        # Build TE maps using only the training fold
        global_mean = float(temp_train_fe["Survived"].mean())
        te_maps = {}
        for col in te_cols:
            stats = temp_train_fe.groupby(col)["Survived"].agg(["sum", "count"])
            te = (stats["sum"] + alpha * global_mean) / (stats["count"] + alpha)
            te_maps[col] = te


        train_fold_fe, fare_bins, age_edges = engineer_features(
            train.iloc[tr_idx], fare_bins=fare_bins, age_edges=age_edges, survival_map=survival_map, 
            te_maps=te_maps, global_mean=global_mean   
        )
        valid_fold_fe, _, _ = engineer_features(
            train.iloc[va_idx], fare_bins=fare_bins, age_edges=age_edges, survival_map=survival_map,
            te_maps=te_maps, global_mean=global_mean 
        )

        # --- Select features ---
        X_tr = train_fold_fe[features].copy()
        y_tr = train.iloc[tr_idx][target].astype(int).values
        X_va = valid_fold_fe[features].copy()
        y_va = train.iloc[va_idx][target].astype(int).values

        # Ensure categorical bins are typed correctly
        for col in ["FareBin", "AgeBin"]:
            if col in X_tr.columns:
                X_tr[col] = X_tr[col].astype("category")
                X_va[col] = X_va[col].astype("category")

        # --- Preprocessing ---
        prep = make_preprocess()
        X_tr = prep.fit_transform(X_tr, y_tr)
        X_va = prep.transform(X_va)

        # ---- GradientBoostingClassifier expects dense arrays ----
        if hasattr(X_tr, "toarray"):
            X_tr = X_tr.toarray()
        if hasattr(X_va, "toarray"):
            X_va = X_va.toarray()

        # ---- Model ----
        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            subsample=subsample,
            max_depth=max_depth,
            max_features=max_features,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            min_impurity_decrease=min_impurity_decrease,  
            max_leaf_nodes=max_leaf_nodes,             
            validation_fraction=validation_fraction,     
            n_iter_no_change=n_iter_no_change,           
            tol=tol,                                      
            random_state=42,
        )

        # Preserve "balanced" behavior via sample weights
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_tr)

        model.fit(X_tr, y_tr, sample_weight=sample_weight)
        va_proba = model.predict_proba(X_va)[:, 1]
        aucs.append(roc_auc_score(y_va, va_proba))
        
        trial.report(np.mean(aucs), step=fold_idx)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        del X_tr, X_va, y_tr, y_va, model
        gc.collect()

    return float(np.mean(aucs))

# Create Optuna study
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_startup_trials=10))
study.optimize(objective, n_trials=50)

# Best parameters
best_params = study.best_params
print("Best parameters:", best_params)
print("Best CV AUC:", study.best_value)

In [ ]:
# Extract bext hyperparameters
best_params = study.best_params
best_n_est = best_params.get("n_estimators", 200)
best_max_d = best_params.get("max_depth", 3)
best_max_f = best_params.get("max_features", None)
best_min_split = best_params.get("min_samples_split", 2)
best_min_leaf = best_params.get("min_samples_leaf", 1)
best_lr = best_params.get("learning_rate", 0.1)
best_subsample = best_params.get("subsample", 1.0)
best_min_impurity = best_params.get("min_impurity_decrease", 0.0)
best_max_leaf_nodes = best_params.get("max_leaf_nodes", None)
best_validation_fraction = best_params.get("validation_fraction", 0.1)
best_n_iter_no_change = best_params.get("n_iter_no_change", 10)
best_tol = best_params.get("tol", 1e-4)

oof_proba = np.zeros(len(train), dtype=float)
test_proba_blend = np.zeros(len(test), dtype=float)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fare_bins, age_edges = None, None

# target-encoding and smoothing strength
te_cols = ["TicketPrefix", "CabinDeck", "Title_Type", "Embarked"] 
alpha = 20 


for fold, (tr_idx, va_idx) in enumerate(cv.split(train, train[target]), 1):
    print(f"Final training - fold {fold}/5")

    # First apply feature engineering to create FamilySize
    temp_train_fe, fare_bins, age_edges = engineer_features(
        train.iloc[tr_idx], fare_bins=fare_bins, age_edges=age_edges
    )

    # Compute survival map from engineered features
    survival_map = temp_train_fe.groupby("FamilySize")["Survived"].mean().to_dict()

    # Build TE maps on the training fold only
    global_mean = float(temp_train_fe["Survived"].mean())
    te_maps = {}
    for col in te_cols:
        stats = temp_train_fe.groupby(col)["Survived"].agg(["sum", "count"])
        te = (stats["sum"] + alpha * global_mean) / (stats["count"] + alpha)
        te_maps[col] = te 


    # Apply feature engineering with survival_map
    train_fold_fe, fare_bins, age_edges = engineer_features(
        train.iloc[tr_idx], fare_bins=fare_bins, age_edges=age_edges, survival_map=survival_map,
        te_maps=te_maps, global_mean=global_mean 
    )
    valid_fold_fe, _, _ = engineer_features(
        train.iloc[va_idx], fare_bins=fare_bins, age_edges=age_edges, survival_map=survival_map,
        te_maps=te_maps, global_mean=global_mean 
    )

    # Apply feature engineering to test set using full survival map
    full_survival_map = temp_train_fe.groupby("FamilySize")["Survived"].mean().to_dict()
    test_fe, _, _ = engineer_features(
        test, fare_bins=fare_bins, age_edges=age_edges, survival_map=full_survival_map,
        te_maps=te_maps, global_mean=global_mean

    )

    # Select features dynamically
    X_tr = train_fold_fe[features].copy()
    y_tr = train.iloc[tr_idx][target].astype(int).values
    X_va = valid_fold_fe[features].copy()
    y_va = train.iloc[va_idx][target].astype(int).values
    X_te = test_fe[features].copy()

    # Convert bin columns to category
    for col in ["FareBin", "AgeBin"]:
        if col in X_tr.columns:
            X_tr[col] = X_tr[col].astype("category")
            X_va[col] = X_va[col].astype("category")
            X_te[col] = X_te[col].astype("category")

    # Preprocessing
    prep = make_preprocess()
    X_tr = prep.fit_transform(X_tr, y_tr)
    X_va = prep.transform(X_va)
    X_te = prep.transform(X_te)

    # ---- NEW: Ensure dense arrays for GradientBoostingClassifier ----
    if hasattr(X_tr, "toarray"):
        X_tr = X_tr.toarray()
    if hasattr(X_va, "toarray"):
        X_va = X_va.toarray()
    if hasattr(X_te, "toarray"):
        X_te = X_te.toarray()

    # ---- CHANGED: Gradient Boosting model (no bootstrap, class_weight, or n_jobs) ----
    model = GradientBoostingClassifier(
        n_estimators=best_n_est,
        learning_rate=best_lr,
        max_depth=best_max_d,
        max_features=best_max_f,
        min_samples_split=best_min_split,
        min_samples_leaf=best_min_leaf,
        subsample=best_subsample,
        min_impurity_decrease=best_min_impurity,       # NEW
        max_leaf_nodes=best_max_leaf_nodes,           # NEW
        validation_fraction=best_validation_fraction, # NEW
        n_iter_no_change=best_n_iter_no_change,       # NEW
        tol=best_tol,                                 # NEW
        random_state=42,
    )

    # ---- OPTIONAL: preserve "balanced" behavior via sample_weight ----
    sample_weight = compute_sample_weight(class_weight="balanced", y=y_tr)

    # Fit model
    model.fit(X_tr, y_tr, sample_weight=sample_weight)
    oof_proba[va_idx] = model.predict_proba(X_va)[:, 1]
    test_proba_blend += model.predict_proba(X_te)[:, 1] / cv.n_splits

    # SHAP analysis
    if fold == 1:
        import shap
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_va)
        shap.summary_plot(shap_values, X_va, feature_names=prep.get_feature_names_out())

        # Low-impact feature removal
        feature_names = prep.get_feature_names_out()
        importances = model.feature_importances_
        feat_imp = pd.DataFrame({"feature": feature_names, "importance": importances})
        threshold = 0.001
        low_impact_features = feat_imp[feat_imp["importance"] < threshold]["feature"].tolist()
        print(f"Dropping {len(low_impact_features)} low-impact features: {low_impact_features}")

    del X_tr, X_va, X_te, y_tr, y_va, model
    gc.collect()

# Tune threshold
grid = np.linspace(0.3, 0.7, 81)
accs = [accuracy_score(train[target], (oof_proba >= t).astype(int)) for t in grid]
best_thr = float(grid[int(np.argmax(accs))])
oof_acc = max(accs)
oof_auc = roc_auc_score(train[target], oof_proba)
print(f"OOF AUC={oof_auc:.4f}, OOF ACC={oof_acc:.4f} @ thr={best_thr:.3f}")

In [ ]:
# Compute metrics
fpr, tpr, _ = roc_curve(train[target], oof_proba)
roc_auc = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(train[target], oof_proba)
avg_precision = average_precision_score(train[target], oof_proba)
prob_true, prob_pred = calibration_curve(train[target], oof_proba, n_bins=10)
y_pred = (oof_proba >= best_thr).astype(int)
cm = confusion_matrix(train[target], y_pred)

# Create plot
fig, axs = plt.subplots(2, 2, figsize=(14, 10))


# ROC Curve
axs[0, 0].plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
axs[0, 0].plot([0, 1], [0, 1], 'k--')
axs[0, 0].set_title('ROC Curve')
axs[0, 0].set_xlabel('False Positive Rate')
axs[0, 0].set_ylabel('True Positive Rate')
axs[0, 0].legend()
axs[0, 0].grid(True)

# Precision-Recall Curve
axs[0, 1].plot(recall, precision, label=f'AP = {avg_precision:.2f}')
axs[0, 1].set_title('Precision-Recall Curve')
axs[0, 1].set_xlabel('Recall')
axs[0, 1].set_ylabel('Precision')
axs[0, 1].legend()
axs[0, 1].grid(True)

# Calibration Curve
axs[1, 0].plot(prob_pred, prob_true, marker='o', label='Calibration')
axs[1, 0].plot([0, 1], [0, 1], 'k--', label='Perfect')
axs[1, 0].set_title('Calibration Curve')
axs[1, 0].set_xlabel('Mean Predicted Probability')
axs[1, 0].set_ylabel('Fraction of Positives')
axs[1, 0].legend()
axs[1, 0].grid(True)

# Confusion Matrix
ConfusionMatrixDisplay(cm).plot(ax=axs[1, 1], cmap='Blues')
axs[1, 1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# Prepare log dictionary using your computed values
final_log = {
    "best_params": {
        "n_estimators": best_n_est,
        "max_depth": best_max_d,
        "max_features": best_max_f,
        "min_samples_split": best_min_split,
        "min_samples_leaf": best_min_leaf,
        "learning_rate": best_lr,
        "subsample": best_subsample
    },
    "threshold": best_thr,
    "oof_auc": oof_auc,
    "oof_acc": oof_acc,
    "dropped_features": low_impact_features
}

# Save to JSON file
with open("final_model_log.json", "w") as f:
    json.dump(final_log, f, indent=4)

print("Final model training results saved to final_model_log.json")

In [ ]:
# Bagged predictions -> submission
test_pred = (test_proba_blend >= best_thr).astype(int)

submission = pd.DataFrame({
    "PassengerId": test_fe["PassengerId"].values,
    "Survived": test_pred
})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")